# SHAP Figures – Exploratory Notebook

Interactive companion to `scripts/shap_figures.py`.
Loads pre-computed SHAP `.npy` arrays and lets you adjust
normalisation, thresholds, and layout interactively.

**Prerequisites:** run `scripts/shap_regression.py` first.

In [ ]:
from __future__ import annotations
import glob, json, os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.colors import ListedColormap
from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

## 1. Configuration

In [ ]:
MODEL_CONFIG      = "./config/model.json"
CASE_STUDIES_FILE = "./config/case_studies.json"
SHAP_DIR          = "./shap/mod512"
VAR               = "msl"   # z500 | peva | msl | sm

with open(MODEL_CONFIG) as f:
    cfg = json.load(f)
with open(CASE_STUDIES_FILE) as f:
    cs = json.load(f)

domain     = cfg["domain"]
hw_cases   = cs["hw"]
nohw_cases = cs["no_hw"]
print(f"{len(hw_cases)} HW and {len(nohw_cases)} NO-HW case studies.")

## 2. Build Colormaps & Geographic Mesh

In [ ]:
N_COLORS = 21
discrete_seismic = ListedColormap(
    plt.get_cmap("RdBu_r")(np.linspace(0, 1, N_COLORS)))
discrete_pg = ListedColormap(
    plt.get_cmap("PiYG")(np.linspace(0, 1, N_COLORS)))
cw = ListedColormap(["none", "none"])

lon_range = np.arange(domain["longitude_min"],
                      domain["longitude_max"] + domain["resolution"],
                      domain["resolution"])
lat_range = np.arange(domain["latitude_min"],
                      domain["latitude_max"] + domain["resolution"],
                      domain["resolution"])
X, Y = np.meshgrid(lon_range, lat_range)
print(f"Grid: lon {lon_range[0]}..{lon_range[-1]},  lat {lat_range[0]}..{lat_range[-1]}")

## 3. Load SHAP Arrays

In [ ]:
pattern = os.path.join(SHAP_DIR, "cs_*", f"shap_*_meansum_{VAR}*.npy")
files   = sorted(glob.glob(pattern))
print(f"{len(files)} files found for variable {VAR!r}")

# Split by time-period (all/post/pre interleaved every 3)
paths_all, paths_post, paths_pre = files[0::3], files[1::3], files[2::3]

# Load the "all" period for quick exploration
raw     = np.array([np.flip(np.load(p)[0].reshape(20, 30), axis=0)
                    for p in paths_all])
ms_max  = float(np.max(np.abs(raw)))
ms_shap = raw / ms_max

n_cs    = ms_shap.shape[0] // 2
ms_shap = ms_shap.reshape(2, n_cs, 20, 30)   # (HW/NOHW, n_cs, lat, lon)
print(f"ms_shap shape: {ms_shap.shape}")

## 4. Quick Visualisation – Mean SHAP per Event Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
titles = ["HW Mean", "NO-HW Mean"]
mean_shap = ms_shap.mean(axis=1)   # (2, lat, lon)

for ax, data, title in zip(axes, mean_shap, titles):
    m = Basemap(
        ax=ax, resolution="l",
        llcrnrlon=domain["longitude_min"], llcrnrlat=domain["latitude_min"],
        urcrnrlon=domain["longitude_max"], urcrnrlat=domain["latitude_max"],
    )
    m.drawcountries(color="#303338")
    m.drawcoastlines(color="#000000")
    img = m.imshow(data, cmap=discrete_seismic, vmin=-1, vmax=1)
    m.drawmeridians(range(0, 360, 10), color="k", labels=[0,0,0,1])
    m.drawparallels(range(-90, 100, 10), color="k", labels=[1,0,0,0])
    m.colorbar(img, extend="both", location="bottom", pad=0.3)
    ax.set_title(f"{title} – {VAR.upper()}", fontsize=13)

fig.tight_layout()
plt.show()

## 5. Individual Case Study Comparison

In [ ]:
CS_IDX = 0   # change to inspect a different case study (0-based)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (i, event_type) in zip(axes, enumerate(["hw", "nohw"])):
    cases = hw_cases if i == 0 else nohw_cases
    label = cases[CS_IDX]["label"]
    data  = ms_shap[i, CS_IDX]

    m = Basemap(
        ax=ax, resolution="l",
        llcrnrlon=domain["longitude_min"], llcrnrlat=domain["latitude_min"],
        urcrnrlon=domain["longitude_max"], urcrnrlat=domain["latitude_max"],
    )
    m.drawcountries(color="#303338")
    m.drawcoastlines(color="#000000")
    img = m.imshow(data, cmap=discrete_seismic, vmin=-1, vmax=1)
    m.drawmeridians(range(0, 360, 10), color="k", labels=[0,0,0,1])
    m.drawparallels(range(-90, 100, 10), color="k", labels=[1,0,0,0])
    m.colorbar(img, extend="both", location="bottom", pad=0.3)
    ax.set_title(f"{event_type.upper()} {CS_IDX+1} – {label}", fontsize=11)

fig.suptitle(f"SHAP – {VAR.upper()} – Case Study {CS_IDX+1}", fontsize=13)
fig.tight_layout()
plt.show()

## 6. Batch Figure Generation

For production figures use the command-line script:



All generated figures are saved under .